In [1]:
# Celda 1 — imports
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, BRONZE_DB, SILVER_DB

In [ ]:
def cargar_a_silver(nombre_tabla, query):
    print(f"Leyendo bronze.raw_{nombre_tabla}...")
    rows, cols = CH.execute(query, with_column_types=True)
    df = pd.DataFrame(rows, columns=[c[0] for c in cols])

    df['_processed_at'] = datetime.now()  # agregar aquí
    df = df.where(pd.notnull(df), None)

    CH.execute(f"TRUNCATE TABLE {SILVER_DB}.stg_{nombre_tabla}")
    CH.execute(f"INSERT INTO {SILVER_DB}.stg_{nombre_tabla} VALUES", df.to_dict('records'))
    print(f"✓ {len(df):,} filas cargadas en silver.stg_{nombre_tabla}")

In [3]:
# Celda 3 — definición de tablas y transformaciones como SQL
tablas = {
    'customers': f"""
        SELECT
            customer_id,
            trimBoth(company_name)                    AS company_name,
            trimBoth(ifNull(contact_name,  ''))       AS contact_name,
            trimBoth(ifNull(contact_title, ''))       AS contact_title,
            trimBoth(ifNull(address,       ''))       AS address,
            trimBoth(ifNull(city,          ''))       AS city,
            trimBoth(ifNull(region,        ''))       AS region,
            trimBoth(ifNull(postal_code,   ''))       AS postal_code,
            trimBoth(ifNull(country,       ''))       AS country,
            trimBoth(ifNull(phone,         ''))       AS phone,
            trimBoth(ifNull(fax,           ''))       AS fax
        FROM {BRONZE_DB}.raw_customers
    """,

    'orders': f"""
        SELECT
            order_id,
            ifNull(customer_id,  '')                  AS customer_id,
            ifNull(employee_id,  0)                   AS employee_id,
            order_date,
            required_date,
            shipped_date,
            ifNull(ship_via,     0)                   AS ship_via,
            ifNull(freight,      0.0)                 AS freight,
            trimBoth(ifNull(ship_name,         ''))   AS ship_name,
            trimBoth(ifNull(ship_address,      ''))   AS ship_address,
            trimBoth(ifNull(ship_city,         ''))   AS ship_city,
            trimBoth(ifNull(ship_region,       ''))   AS ship_region,
            trimBoth(ifNull(ship_postal_code,  ''))   AS ship_postal_code,
            trimBoth(ifNull(ship_country,      ''))   AS ship_country
        FROM {BRONZE_DB}.raw_orders
        WHERE order_date IS NOT NULL
    """,

    'order_details': f"""
        SELECT
            order_id,
            product_id,
            ifNull(unit_price, 0.0) AS unit_price,
            ifNull(quantity,   0)   AS quantity,
            ifNull(discount,   0.0) AS discount
        FROM {BRONZE_DB}.raw_order_details
    """,

    'products': f"""
        SELECT
            product_id,
            trimBoth(ifNull(product_name,      ''))  AS product_name,
            ifNull(supplier_id,  0)                  AS supplier_id,
            ifNull(category_id,  0)                  AS category_id,
            trimBoth(ifNull(quantity_per_unit, ''))  AS quantity_per_unit,
            ifNull(unit_price,   0.0)                AS unit_price,
            ifNull(units_in_stock, 0)                AS units_in_stock,
            ifNull(units_on_order, 0)                AS units_on_order,
            ifNull(reorder_level,  0)                AS reorder_level,
            ifNull(discontinued,   0)                AS discontinued
        FROM {BRONZE_DB}.raw_products
    """,

    'categories': f"""
        SELECT
            category_id,
            trimBoth(ifNull(category_name, '')) AS category_name,
            trimBoth(ifNull(description,   '')) AS description
        FROM {BRONZE_DB}.raw_categories
    """,

    'suppliers': f"""
        SELECT
            supplier_id,
            trimBoth(ifNull(company_name,  '')) AS company_name,
            trimBoth(ifNull(contact_name,  '')) AS contact_name,
            trimBoth(ifNull(contact_title, '')) AS contact_title,
            trimBoth(ifNull(address,       '')) AS address,
            trimBoth(ifNull(city,          '')) AS city,
            trimBoth(ifNull(region,        '')) AS region,
            trimBoth(ifNull(postal_code,   '')) AS postal_code,
            trimBoth(ifNull(country,       '')) AS country,
            trimBoth(ifNull(phone,         '')) AS phone,
            trimBoth(ifNull(fax,           '')) AS fax,
            trimBoth(ifNull(homepage,      '')) AS homepage
        FROM {BRONZE_DB}.raw_suppliers
    """,

    'employees': f"""
        SELECT
            employee_id,
            trimBoth(ifNull(last_name,          '')) AS last_name,
            trimBoth(ifNull(first_name,         '')) AS first_name,
            concat(
                trimBoth(ifNull(first_name, '')),
                ' ',
                trimBoth(ifNull(last_name,  ''))
            )                                        AS full_name,
            trimBoth(ifNull(title,              '')) AS title,
            trimBoth(ifNull(title_of_courtesy,  '')) AS title_of_courtesy,
            birth_date,
            hire_date,
            trimBoth(ifNull(address,            '')) AS address,
            trimBoth(ifNull(city,               '')) AS city,
            trimBoth(ifNull(region,             '')) AS region,
            trimBoth(ifNull(postal_code,        '')) AS postal_code,
            trimBoth(ifNull(country,            '')) AS country,
            trimBoth(ifNull(home_phone,         '')) AS home_phone,
            trimBoth(ifNull(extension,          '')) AS extension,
            trimBoth(ifNull(notes,              '')) AS notes
        FROM {BRONZE_DB}.raw_employees
    """,

    'shippers': f"""
        SELECT
            shipper_id,
            trimBoth(ifNull(company_name, '')) AS company_name,
            trimBoth(ifNull(phone,        '')) AS phone
        FROM {BRONZE_DB}.raw_shippers
    """,

    'territories': f"""
        SELECT
            territory_id,
            trimBoth(ifNull(territory_description, '')) AS territory_description,
            ifNull(region_id, 0)                        AS region_id
        FROM {BRONZE_DB}.raw_territories
    """,

    'region': f"""
        SELECT
            region_id,
            trimBoth(ifNull(region_description, '')) AS region_description
        FROM {BRONZE_DB}.raw_region
    """,
}

In [8]:
# Celda 4 — ejecutar
for tabla, query in tablas.items():
    cargar_a_silver(tabla, query)

Leyendo bronze.raw_customers...
✓ 91 filas cargadas en silver.stg_customers
Leyendo bronze.raw_orders...
✓ 830 filas cargadas en silver.stg_orders
Leyendo bronze.raw_order_details...
✓ 2,155 filas cargadas en silver.stg_order_details
Leyendo bronze.raw_products...
✓ 77 filas cargadas en silver.stg_products
Leyendo bronze.raw_categories...
✓ 8 filas cargadas en silver.stg_categories
Leyendo bronze.raw_suppliers...
✓ 29 filas cargadas en silver.stg_suppliers
Leyendo bronze.raw_employees...
✓ 9 filas cargadas en silver.stg_employees
Leyendo bronze.raw_shippers...
✓ 6 filas cargadas en silver.stg_shippers
Leyendo bronze.raw_territories...
✓ 53 filas cargadas en silver.stg_territories
Leyendo bronze.raw_region...
✓ 4 filas cargadas en silver.stg_region
